# Project 2 — Offline API Client and ETL Pipeline

Simulate a real-world HTTP data ingestion workflow without making network calls. Create a fake HTTP transport, validate JSON, traverse pages, reject malformed records, compute aggregates, and export a CSV report.

**Level:** Intermediate project · **Suggested time:** 3–5 hours · **Prerequisites:** Intermediate lessons 00–19.

## Project requirements

- Build a testable HTTP-style transport with explicit timeout
- Parse and validate API product records
- Handle pagination loops and duplicate products
- Aggregate normalized results and export CSV
- Test error behavior without live network

**Instructions:** Study the explanations and runnable implementation, then complete the extension challenges. All examples use the Python standard library and offline data. Run the notebook from a fresh kernel to test reproducibility.

## 1. Domain model and validation

API data is untrusted even when JSON syntax is valid. A frozen Product validates required strings and rejects negative or non-finite numeric prices. Type annotations alone do not protect runtime values.

In [ ]:
from dataclasses import dataclass
import math

@dataclass(frozen=True)
class Product:
    sku: str
    category: str
    price: float
    def __post_init__(self):
        if not isinstance(self.sku, str) or not self.sku.strip():
            raise ValueError("invalid SKU")
        if not isinstance(self.category, str) or not self.category.strip():
            raise ValueError("invalid category")
        if isinstance(self.price, bool) or not isinstance(self.price, (int, float)) or not math.isfinite(self.price) or self.price < 0:
            raise ValueError("invalid price")

def parse_product(raw):
    if not isinstance(raw, dict):
        raise ValueError("record is not an object")
    return Product(raw["sku"], raw["category"], raw["price"])

print(parse_product({"sku": "P1", "category": "books", "price": 10.5}))

**Design question:** What is the consequence of accepting NaN as a price?

## 2. Fake HTTP response and JSON decoding

A transport accepts a Request and timeout and returns bytes, just like a real HTTP client would. A fake opener is deterministic and does not need credentials. The adapter handles HTTP/JSON details and is separate from business validation.

In [ ]:
import json
from io import BytesIO
from urllib.request import Request

PAGES = {
    1: {"items": [{"sku": "P1", "category": "books", "price": 12.5}], "next": 2},
    2: {"items": [{"sku": "P2", "category": "pens", "price": 2.0}], "next": None},
}

def fake_open(request, timeout):
    assert request.get_method() == "GET" and timeout > 0
    page = int(request.full_url.split("page=")[-1])
    return BytesIO(json.dumps(PAGES[page]).encode("utf-8"))

def fetch_page(page, opener=fake_open, timeout=5):
    request = Request(f"https://example.invalid/api/products?page={page}", method="GET")
    with opener(request, timeout=timeout) as response:
        payload = json.loads(response.read().decode("utf-8"))
    if not isinstance(payload, dict) or not isinstance(payload.get("items"), list):
        raise ValueError("invalid response schema")
    return payload

print(fetch_page(1))

**Design question:** Why is injecting an opener better than patching a global network function everywhere?

## 3. Bounded ingestion with error report

The pipeline tracks seen pages and SKUs. A page limit avoids infinite loops and a rejection log allows the caller to inspect individual bad records rather than losing all valid data. The policy here keeps the first valid SKU.

In [ ]:
def collect_products(fetch, first=1, max_pages=10):
    visited, skus = set(), set()
    products, errors = [], []
    page = first
    while page is not None:
        if page in visited or len(visited) >= max_pages:
            raise RuntimeError("pagination loop or maximum reached")
        visited.add(page)
        response = fetch(page)
        for index, raw in enumerate(response["items"], start=1):
            try:
                product = parse_product(raw)
                if product.sku in skus:
                    raise ValueError("duplicate SKU")
                skus.add(product.sku)
                products.append(product)
            except (ValueError, KeyError, TypeError) as error:
                errors.append((page, index, str(error)))
        page = response.get("next")
    return products, errors

products, rejected = collect_products(fetch_page)
print(products, rejected)
assert len(products) == 2

**Design question:** How does page+index help someone investigate a failed import?

## 4. Aggregation and CSV export

Keep transformations pure and export output using csv.DictWriter. StringIO produces a report in memory without modifying the repository; a real application can write the text to a user-selected path.

In [ ]:
from collections import defaultdict
from io import StringIO
import csv

def category_totals(products):
    totals = defaultdict(float)
    for product in products:
        totals[product.category] += product.price
    return dict(sorted(totals.items()))

def export_csv(totals):
    destination = StringIO()
    writer = csv.DictWriter(destination, fieldnames=["category", "total"])
    writer.writeheader()
    for category, total in totals.items():
        writer.writerow({"category": category, "total": f"{total:.2f}"})
    return destination.getvalue()

print(export_csv(category_totals(products)))

**Design question:** Does this report represent inventory value or revenue? What additional data would you need?

## 5. End-to-end tests

Replace the fetch function with deterministic fake pages to verify duplicates and loops. No external call is made, so tests run offline without flaky remote dependencies.

In [ ]:
import unittest

class ApiPipelineTests(unittest.TestCase):
    def test_two_pages(self):
        self.assertEqual(len(collect_products(fetch_page)[0]), 2)
    def test_duplicate(self):
        def duplicate(_):
            item = {"sku": "P1", "category": "books", "price": 10}
            return {"items": [item, item], "next": None}
        accepted, rejected = collect_products(duplicate)
        self.assertEqual((len(accepted), len(rejected)), (1, 1))
    def test_cycle(self):
        with self.assertRaises(RuntimeError):
            collect_products(lambda page: {"items": [], "next": page})

assert unittest.TextTestRunner(verbosity=0).run(
    unittest.defaultTestLoader.loadTestsFromTestCase(ApiPipelineTests)
).wasSuccessful()

**Design question:** What exception types might a real HTTP opener raise?

## Independent extension challenges

Add features in small increments. For each change, decide how to validate input and how to test failures.

**Challenge 1:** Add a bounded retry policy for transient GET failures, with an injectable sleep function.

In [ ]:
# TODO: Write code and assertions for challenge 1.

**Challenge 2:** Follow token-based next-page responses and reject repeated tokens.

In [ ]:
# TODO: Write code and assertions for challenge 2.

**Challenge 3:** Write invalid records into a separate CSV rejection report.

In [ ]:
# TODO: Write code and assertions for challenge 3.

**Challenge 4:** Benchmark SKU deduplication using a list versus a set, and explain trade-offs.

In [ ]:
# TODO: Write code and assertions for challenge 4.

## Portfolio quality checklist

- [ ] The implementation separates domain logic from I/O.
- [ ] There are tests for both normal and invalid inputs.
- [ ] Data validation is explicit and error messages are useful.
- [ ] Resources are closed and examples are safe to rerun.
- [ ] No live API calls or credentials are necessary to test.
- [ ] A README explains the project design, assumptions, and usage.